<a href="https://colab.research.google.com/github/MayaHayat/try/blob/main/Copy_of_Machine_learning__EX3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from collections import Counter


In [ ]:
# Initialize a list to store all rows
x_data = []

# Open the file for reading
with open('vectors.txt', 'r') as file:
    # Iterate over each line in the file
    for line in file:
        # Strip leading/trailing whitespace and split the line by spaces
        parts = line.strip().split()
        # Convert parts to integers and append to the list of rows
        row = [int(part) for part in parts[:8]]
        x_data.append(row)

# Convert the list of rows to a numpy array
x_rows = np.array(x_data)

# Print the resulting array
print(x_rows[0])


[0 1 0 1 1 0 0 0]


In [ ]:
# Initialize a list to store all values
y_data = []

# Open the file for reading
with open('vectors.txt', 'r') as file:
    # Iterate over each line in the file
    for line in file:
        # Strip leading/trailing whitespace and split the line by spaces
        parts = line.strip().split()
        # Convert parts to integers and append to the list of rows
        last_number = int(parts[-1])
        y_data.append(last_number)

# Convert the list of rows to a numpy array
y_rows = np.array(y_data)

# Print the resulting array
print(y_rows)

[1 0 1 0 0 1 0 0 0 1 0 1 0 0 0 0 1 0 1 0 0 1 1 0 1 1 1 1 0 0 1 0 0 1 1 1 0
 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 1 1 0 0 0 1 1 1 0 0 0 0 0 1 1 1 0 0 0 1
 1 1 1 0 1 0 1 0 1 1 0 1 1 1 0 1 1 1 1 1 0 1 1 0 1 0 0 1 1 1 0 1 0 1 0 1 0
 0 1 1 1 1 0 1 0 1 1 0 1 1 0 0 0 1 1 0 1 1 0 0 1 1 1 0 1 0 0 1 0 0 0 1 0 0
 0 1]


In [ ]:
# Level 1 : No features

# Level 2 : 7 choose 1

# Level 3 : 7 choose 1 for root * 6 * 5

# Level 4 : 7 * 6 * 5 * 4 on left or on right + 7 * 6 * 5 * 4 * 3 * 2

# Part A _ Brute Force

In [ ]:
def compute_error(tree, y_rows, predicted):
    if (len(y_rows) != len(predicted)):
        raise ValueError("Predicted and expected must have the same length.")
    num_errors = 0
    for i in range (len(y_rows)):
        if predicted[i] != y_rows[i]:
            num_errors += 1
    return num_errors

In [ ]:
from itertools import product

def generate_leaf_combinations(num_leaves):
    # Generate all possible combinations of 0s and 1s for num_leaves leaves
    return list(product([0, 1], repeat=num_leaves))

def generate_trees(depth, dimensions):
    # Recursive function to generate all trees of given depth
    def generate_node(depth):
        num_leaves = 2 ** depth
        if depth == 0:
            # For leaves, generate all combinations of 0s and 1s
            return [tuple(comb) for comb in generate_leaf_combinations(num_leaves)]
        else:
            trees = set()
            for dim in range(dimensions):
                left_subtrees = generate_node(depth - 1)
                right_subtrees = generate_node(depth - 1)
                for left in left_subtrees:
                    for right in right_subtrees:
                        tree = (dim, left, right)
                        trees.add(prune_tree(tree))  # Add the pruned tree to the set
            return trees

    return generate_node(depth)

def prune_tree(tree):
    # Recursive function to prune identical left and right subtrees
    if isinstance(tree, tuple) and len(tree) == 3:
        dim, left, right = tree
        left = prune_tree(left)
        right = prune_tree(right)
        if left == right:
            # If both subtrees are the same, turn the node into that subtree
            return left
        return (dim, left, right)
    elif isinstance(tree, tuple):
        # This is a leaf node, just return it
        return tree

def format_tree(tree):
    # Recursive function to format the tree
    if isinstance(tree, tuple) and len(tree) == 3:
        dim, left, right = tree
        left_str = format_tree(left)
        right_str = format_tree(right)
        return f"{{{dim} {{left : {left_str} , right : {right_str}}}}}"
    elif isinstance(tree, tuple):
        # This is a leaf node, format it without commas
        return f"Leaf : {''.join(map(str, tree))}"

def print_trees(trees):
    for i, tree in enumerate(trees):
        formatted_tree = format_tree(tree)
        print(f"Tree {i + 1}: {formatted_tree}")




In [ ]:
def predict_tree(tree, vector):
    if isinstance(tree, tuple) and len(tree) == 3:
        dim, left, right = tree
        if vector[dim] == 0:
            return predict_tree(left, vector)
        else:
            return predict_tree(right, vector)
    else:
        return tree

def compute_error(y_rows, predicted):
    if len(y_rows) != len(predicted):
        raise ValueError("Predicted and expected must have the same length.")
    num_errors = 0
    for i in range(len(y_rows)):
        if predicted[i] != y_rows[i]:
            num_errors += 1
    return num_errors

def evaluate_tree(tree, vectors, labels):
    predictions = [predict_tree(tree, vector) for vector in vectors]
    return compute_error(labels, predictions)

def compute_best_tree(trees, vectors, labels):
    best_tree = None
    best_error = float('inf')  # Initialize with infinity to find the minimum
    for tree in trees:
        error = evaluate_tree(tree, vectors, labels)
        formatted_tree = format_tree(tree)
        print(f"Error: {error}, Tree: {formatted_tree}")
        if error < best_error:
            best_error = error
            best_tree = tree
    return best_tree, best_error


In [ ]:
k = 3
depth = k-1
dimensions = len(x_rows[0])
trees = generate_trees(depth, dimensions)
vectors = x_rows
labels = y_rows

best_tree, best_error= compute_best_tree(trees, vectors, labels)

print(f"Best Tree: {best_tree}")
print(f"Best Error: {best_error}")


Error: 72, Tree: {3 {left : {0 {left : Leaf : 1 , right : Leaf : 0}} , right : Leaf : 0}}
Error: 91, Tree: {1 {left : {0 {left : Leaf : 0 , right : Leaf : 1}} , right : {4 {left : Leaf : 0 , right : Leaf : 1}}}}
Error: 66, Tree: {7 {left : {5 {left : Leaf : 1 , right : Leaf : 0}} , right : {6 {left : Leaf : 1 , right : Leaf : 0}}}}
Error: 79, Tree: {6 {left : {2 {left : Leaf : 0 , right : Leaf : 1}} , right : {1 {left : Leaf : 0 , right : Leaf : 1}}}}
Error: 79, Tree: {0 {left : Leaf : 0 , right : {6 {left : Leaf : 1 , right : Leaf : 0}}}}
Error: 77, Tree: {6 {left : {6 {left : Leaf : 0 , right : Leaf : 1}} , right : {3 {left : Leaf : 1 , right : Leaf : 0}}}}
Error: 74, Tree: {6 {left : {2 {left : Leaf : 1 , right : Leaf : 0}} , right : {6 {left : Leaf : 0 , right : Leaf : 1}}}}
Error: 66, Tree: {4 {left : {7 {left : Leaf : 0 , right : Leaf : 1}} , right : {1 {left : Leaf : 1 , right : Leaf : 0}}}}
Error: 74, Tree: {2 {left : Leaf : 0 , right : {2 {left : Leaf : 1 , right : Leaf : 0}}}

# Part B _ Binary Entropy

In [ ]:
import numpy as np
import math

def entropy(data):
    labels = data[:, -1]
    unique_labels, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)
    return -np.sum([p * np.log2(p) for p in probabilities])

def information_gain(data, feature):
    # Calculate the entropy before the split
    original_entropy = entropy(data)

    # Split the data based on the feature
    left_data, right_data = split_data(data, feature)

    # Calculate the entropy after the split
    if len(left_data) == 0 or len(right_data) == 0:
        return 0

    left_entropy = entropy(left_data)
    right_entropy = entropy(right_data)

    # Calculate the weighted entropy
    p_left = len(left_data) / len(data)
    p_right = len(right_data) / len(data)
    weighted_entropy = p_left * left_entropy + p_right * right_entropy

    # Calculate the information gain
    return original_entropy - weighted_entropy

def split_data(data, feature):
    threshold = np.median(data[:, feature])
    left_data = data[data[:, feature] <= threshold]
    right_data = data[data[:, feature] > threshold]
    return left_data, right_data

def build_tree(data, depth, max_depth):
    # Base cases:
    if depth == max_depth or len(data) == 0 or len(set(data[:, -1])) == 1:
        # Create a leaf node
        return {'label': int(np.argmax(np.bincount(data[:, -1].astype(int))))}

    # Find the best split
    best_gain = -1
    best_feature = None
    for feature in range(data.shape[1] - 1):
        gain = information_gain(data, feature)
        if gain > best_gain:
            best_gain = gain
            best_feature = feature

    # Split the data based on the best feature
    left_data, right_data = split_data(data, best_feature)

    # Recursively build subtrees
    left_subtree = build_tree(left_data, depth + 1, max_depth)
    right_subtree = build_tree(right_data, depth + 1, max_depth)

    return {'feature': best_feature, 'left': left_subtree, 'right': right_subtree}

def format_tree(tree):
    if 'label' in tree:
        return f"Leaf : {tree['label']}"
    else:
        feature = tree['feature']
        left_str = format_tree(tree['left'])
        right_str = format_tree(tree['right'])
        return f"{{{feature} {{left : {left_str} , right : {right_str}}}}}"


# Example usage
np.random.seed(0)
X = x_rows
y = y_rows
data = np.column_stack((X, y))
max_depth = 2

tree_entropy = build_tree(data, 0, max_depth)
print(format_tree(tree_entropy))


{5 {left : {6 {left : Leaf : 0 , right : Leaf : 1}} , right : {7 {left : Leaf : 0 , right : Leaf : 1}}}}


In [ ]:
def predict_tree(tree, vector):
    if isinstance(tree, tuple) and len(tree) == 3:
        dim, left, right = tree
        if vector[dim] == 0:
            return predict_tree(left, vector)
        else:
            return predict_tree(right, vector)
    else:
        return tree

def compute_error(y_rows, predicted):
    if len(y_rows) != len(predicted):
        raise ValueError("Predicted and expected must have the same length.")
    num_errors = 0
    for i in range(len(y_rows)):
        if predicted[i] != y_rows[i]:
            num_errors += 1
    return num_errors

def evaluate_tree(tree, vectors, labels):
    predictions = [predict_tree(tree, vector) for vector in vectors]
    return compute_error(labels, predictions)